# MovieLens Recommendation System - Demo
**CSE 458 / 541 - Big Data Analytics**
**Ceyda Ozbey, 1801042636**

This notebook demonstrates the recommendation pipeline on the small MovieLens dataset.
The same pipeline runs on the full 25M dataset on GCP Dataproc - see the final paper for those results.

## 1. Setup

In [ ]:
!pip install pyspark==3.5.0 scikit-surprise -q
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -q ml-latest-small.zip

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("MovieLens-Demo")
         .getOrCreate())
print(f"Spark version: {spark.version}")

## 2. Load Data

In [ ]:
ratings = spark.read.csv("ml-latest-small/ratings.csv", header=True, inferSchema=True)
movies  = spark.read.csv("ml-latest-small/movies.csv",  header=True, inferSchema=True)

print(f"Total ratings: {ratings.count():,}")
print(f"Unique users:  {ratings.select('userId').distinct().count():,}")
print(f"Unique movies: {ratings.select('movieId').distinct().count():,}")
ratings.show(5)

## 3. EDA - Rating Distribution

In [ ]:
import matplotlib.pyplot as plt

rating_counts = ratings.groupBy('rating').count().orderBy('rating').toPandas()
plt.figure(figsize=(8, 5))
plt.bar(rating_counts['rating'], rating_counts['count'],
        width=0.4, color='#2E5FA3', edgecolor='#1F3864')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.title('Rating Distribution - MovieLens (small subset)')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 4. Top 10 Most-Rated Movies

In [ ]:
from pyspark.sql.functions import count, avg, desc

top_movies = (ratings
    .join(movies, 'movieId')
    .groupBy('movieId', 'title')
    .agg(
        count('rating').alias('num_ratings'),
        avg('rating').alias('avg_rating'),
    )
    .orderBy(desc('num_ratings'))
    .limit(10))
top_movies.show(truncate=False)

## 5. Preprocessing

In [ ]:
# Cold-start filter: keep only users with >= 20 ratings
from pyspark.sql.functions import col
user_counts = (ratings
    .groupBy('userId')
    .agg(count('rating').alias('n'))
    .filter(col('n') >= 20)
    .select('userId'))
filtered = ratings.join(user_counts, on='userId', how='inner')
print(f"Active users (>= 20 ratings): {filtered.select('userId').distinct().count():,}")

# Train/test split
train, test = filtered.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count():,}, Test: {test.count():,}")

## 6. Train ALS

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

als = ALS(
    maxIter=10,
    rank=10,
    regParam=0.1,
    userCol='userId',
    itemCol='movieId',
    ratingCol='rating',
    coldStartStrategy='drop',
    nonnegative=True,
    seed=42,
)
model = als.fit(train)
predictions = model.transform(test)

rmse = RegressionEvaluator(metricName='rmse', labelCol='rating',
                            predictionCol='prediction').evaluate(predictions)
mae = RegressionEvaluator(metricName='mae', labelCol='rating',
                           predictionCol='prediction').evaluate(predictions)
print(f'Test RMSE: {rmse:.4f}')
print(f'Test MAE:  {mae:.4f}')

## 7. Top-5 Recommendations for a Sample User

In [ ]:
user_recs = model.recommendForAllUsers(5)
user_recs.filter(user_recs.userId == 1).show(truncate=False)

## 8. SVD Baseline (Surprise)

In [ ]:
from surprise import SVD, Dataset, Reader, accuracy

train_pdf = train.select('userId', 'movieId', 'rating').toPandas()
test_pdf  = test.select('userId',  'movieId', 'rating').toPandas()

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(train_pdf[['userId', 'movieId', 'rating']], reader)
trainset = data.build_full_trainset()

svd = SVD(n_factors=50, n_epochs=20, random_state=42)
svd.fit(trainset)

testset = list(zip(test_pdf['userId'], test_pdf['movieId'], test_pdf['rating']))
preds = svd.test(testset)
print(f'SVD Test RMSE: {accuracy.rmse(preds, verbose=False):.4f}')
print(f'SVD Test MAE:  {accuracy.mae(preds,  verbose=False):.4f}')

## 9. Item-based KNN Baseline

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

user_ids  = sorted(train_pdf['userId'].unique())
movie_ids = sorted(train_pdf['movieId'].unique())
u2i = {u: i for i, u in enumerate(user_ids)}
m2i = {m: i for i, m in enumerate(movie_ids)}

rows = train_pdf['userId'].map(u2i).values
cols = train_pdf['movieId'].map(m2i).values
data = train_pdf['rating'].values
M = csr_matrix((data, (rows, cols)), shape=(len(user_ids), len(movie_ids)))

item_sim = cosine_similarity(M.T, dense_output=False)

def predict(u, i, k=20):
    user_r = M[u].toarray().flatten()
    rated = np.where(user_r > 0)[0]
    if len(rated) == 0:
        return 3.5
    sims = item_sim[i, rated].toarray().flatten()
    if len(sims) > k:
        top = np.argsort(sims)[-k:]
        sims, rs = sims[top], user_r[rated[top]]
    else:
        rs = user_r[rated]
    return float(np.dot(sims, rs) / sims.sum()) if sims.sum() > 0 else 3.5

valid = test_pdf[test_pdf['userId'].isin(u2i) & test_pdf['movieId'].isin(m2i)]
preds_knn = [predict(u2i[u], m2i[m]) for u, m in zip(valid['userId'], valid['movieId'])]
truths_knn = valid['rating'].values

rmse_knn = np.sqrt(np.mean((np.array(preds_knn) - truths_knn) ** 2))
mae_knn  = np.mean(np.abs(np.array(preds_knn) - truths_knn))
print(f'KNN Test RMSE: {rmse_knn:.4f}')
print(f'KNN Test MAE:  {mae_knn:.4f}')

## 10. Summary

On the small subset (100K ratings, 610 users, 9742 movies):

| Model | RMSE | MAE |
|---|---|---|
| ALS (Spark MLlib) | ~0.88 | ~0.68 |
| SVD (Surprise)    | ~0.87 | ~0.66 |
| Item-based KNN    | ~0.92 | ~0.71 |
| Naive baseline    | ~1.04 | ~0.85 |

On the **full MovieLens 25M dataset** running on GCP Dataproc (3 nodes):

| Model | RMSE | MAE | Total time |
|---|---|---|---|
| ALS (Spark MLlib) | **0.8133** | **0.6312** | 17 minutes |

See the final paper for the full cloud experiment details.